In [63]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision 
from torchvision.datasets import CIFAR10 # Here image is of size 32x32x3 (as it is a coloured image RGB therefore it has 3 channels)

In [64]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#image=> scale(0,1)=> normalize(-1,1)
transform= transforms.Compose([transforms.ToTensor(),  # covert into tensor + scaling
                               transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])  # Normalizing

# CIFAR10 contains training and testing data in it 

trainingset= CIFAR10(root="./data", train=True, download=True, transform=transform)
testingset= CIFAR10(root="./data", train=False, download=True, transform=transform)

In [65]:
trainloader= DataLoader(trainingset, batch_size=64, shuffle=True)
testloader= DataLoader(testingset, batch_size=64, shuffle=True)

In [66]:
# Building  CNN

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1), # i/p channel is 3 (as rgb) and o/p channel is 32 (hyperparameter which gets doubled after every layer)
        nn.ReLU(),
        nn.MaxPool2d(2,2), # Kernel=2, stride=2  (maxpool with these parameter resizes the image size to half i.e now 16)

        nn.Conv2d(32, 64, kernel_size=3, padding=1), 
        nn.ReLU(),
        nn.MaxPool2d(2,2),  # now 8

        nn.Conv2d(64, 128, kernel_size=3, padding=1), 
        nn.ReLU(),
        nn.MaxPool2d(2,2)  # now 4
        )

        self.fc_layers= nn.Sequential(
        nn.Linear(4*4*128, 256), # 4*4*128 is final image size and suppose we take 256 neuron(4x4x128=2048 features which converted into 256)
        nn.ReLU(),

        nn.Linear(256, 10), # and from 256 to 10 classes (there are 10 classes output in CIFAR dataset)
        ) # here we need to apply softmax but as we are using CrossEntropyLoss it already applies Softmax internally.

    def forward(self, x):
        x= self.conv_layers(x)
        x= x.view(x.size(0), -1) # flattening the output
        x= self.fc_layers(x)

        return x

In [67]:
model= CNN()

In [68]:
criteria= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters())

In [69]:
# Training the CNN

epoch=10

for epochs in range(epoch):
    model.train()
    epoch_train_loss= 0.0
    epoch_test_loss= 0.0
    
    for images, labels in trainloader:
        optimizer.zero_grad()  # loss should not accumulate everytime in loop
        output= model(images)  # pforward propagtn
        loss= criteria(output, labels)  # compute loss
        loss.backward()  #back prop.. computing gradients
        optimizer.step()  # updating weight and bias

        epoch_train_loss+= loss.item()  # .item convert tensor value into float
    epoch_train_loss= epoch_train_loss/len(trainloader)

    model.eval()
    with torch.no_grad():
        for images, labels in testloader:
            output= model(images)  # pforward propagtn
            loss= criteria(output, labels)  # compute loss

            epoch_test_loss+= loss.item()  # .item convert tensor value into float
            
    epoch_test_loss= epoch_test_loss/len(testloader)
    print(f"epoch={epochs+1} and train loss={epoch_train_loss} and test loss={epoch_test_loss}")

epoch=1 and train loss=1.3830865084972528 and test loss=1.1098172140728897
epoch=2 and train loss=0.9498811159902216 and test loss=0.8497194674364321
epoch=3 and train loss=0.7566564497740372 and test loss=0.7801236366010775
epoch=4 and train loss=0.6326964376756298 and test loss=0.7582336854023538
epoch=5 and train loss=0.5238293081979313 and test loss=0.7575762782506882
epoch=6 and train loss=0.4384946154282831 and test loss=0.7591898965228135
epoch=7 and train loss=0.357673812933895 and test loss=0.7954828393687109
epoch=8 and train loss=0.28658881566256206 and test loss=0.837834458821898
epoch=9 and train loss=0.2256285488281561 and test loss=0.9248378772264833
epoch=10 and train loss=0.1800011988595852 and test loss=0.9964412593158187


# Model is overfitting so its better to stop at epoch 5

In [70]:
# Evaluation

total = 0
correct = 0

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images) 
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0) # actual samples in each batch

print("accuracy: ", correct/total * 100)

accuracy:  75.86


In [71]:
torch.save(model.state_dict(), "cnn_model.pth")